# Reverse Diffusion — Sample Generation Pipeline

Generates synthetic time-series samples from a trained CSDI checkpoint and saves them to `data/generated/toy/` in the same CSV + stats-pickle format expected by `Toy_dataset_analysis.ipynb`.

**To use in the analysis notebook** update its path variables:
```python
gen_dir       = "../data/generated/toy/"
gen_stats_path = "../data/generated/toy/fake_stats_generated.pkl"
```

## 1. Imports

In [1]:
import os
import sys
import pickle
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import torch
import yaml
from copy import deepcopy

# Make project root importable from the notebooks/ folder
sys.path.insert(0, os.path.abspath(".."))

from src.models.model_core import CSDIModel
from src.utils.WIP_processes import Diffusion_Processes

## 2. Parameters

Edit the three variables below before running the rest of the notebook.

In [2]:
# ── Checkpoint ────────────────────────────────────────────────────────────────
# Name of the .pt file inside  checkpoints/csdi/
CHECKPOINT_FOLDER = "toy"
CHECKPOINT_NAME = "FAKE_final_ep-30_step-92160_sde-ve_lr-1e-04_N-1000_notlinear_layers-4_nheads-8_20260321_050613.pt"

# ── Mask mode ─────────────────────────────────────────────────────────────────
# "unconditional"  → generate all OHLCV features from pure noise
# "predict_close"  → condition on observed O, H, L, V and predict C only
mask_mode = "unconditional"

# ── Generation ────────────────────────────────────────────────────────────────
N_SAMPLES = 100       # number of synthetic time-series to generate

# ── Conditioning dataset (used only when mask_mode == "predict_close") ────────
# Path to a folder containing processed CSV files with columns:
#   Date, Open, High, Low, Close, Volume
# This should be the same dataset used for training.
# N_SAMPLES CSV files will be picked (alphabetically) from this directory.
COND_DATA_DIR = os.path.join("..", "data", "fake_individual_gbm")

# Optional path to a stats pickle {ticker: {"mean": np.array(K,), "std": np.array(K,)}}
# produced during training.  Set to None to compute z-score stats on the fly
# from each individual series.
COND_STATS_PATH = None   # e.g. os.path.join("..", "data", "fake_individual_gbm", "fake_stats.pkl")

# ── Time horizon ──────────────────────────────────────────────────────────────
# Business-day date range used to label the generated rows.
# The range must cover at least seq_len (252) trading days.
START_DATE = "1986-01-01"
END_DATE   = "2025-12-31"

# ── Optional: override the number of reverse diffusion steps ──────────────────
# None  → use the value stored in the checkpoint config (process.N, typically 1000)
# int   → e.g. 200 for a faster (lower quality) run
NUM_REVERSE_STEPS = None

## 3. Load checkpoint and instantiate model

In [3]:
CHECKPOINT_PATH = os.path.join("..", "checkpoints", CHECKPOINT_FOLDER, CHECKPOINT_NAME)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

ckpt   = torch.load(CHECKPOINT_PATH, map_location=device)
config = ckpt["config"]

target_dim = int(config["data"]["target_dim"])   # K = 5 features
seq_len    = int(config["train"]["seq_len"])      # L = 252 time steps

model = CSDIModel(target_dim=target_dim, config=config, device=device).to(device)
model.load_state_dict(ckpt["model"])
model.eval()

print(f"Checkpoint : {CHECKPOINT_NAME}")
print(f"target_dim : {target_dim}  |  seq_len : {seq_len}")
print(f"SDE type   : {config['process']['sde_type']}")

Device: cpu
Checkpoint : FAKE_final_ep-30_step-92160_sde-ve_lr-1e-04_N-1000_notlinear_layers-4_nheads-8_20260321_050613.pt
target_dim : 5  |  seq_len : 252
SDE type   : ve


## 4. Instantiate Diffusion_Processes

In [4]:
processes = Diffusion_Processes(config["process"])

num_reverse_steps = NUM_REVERSE_STEPS if NUM_REVERSE_STEPS is not None else processes.N
print(f"Diffusion_Processes ready — SDE: {processes.sde_type}, N: {processes.N}, model_steps: {processes.model_steps}")
print(f"Reverse steps to use: {num_reverse_steps}")

Diffusion_Processes ready — SDE: ve, N: 1000, model_steps: 50
Reverse steps to use: 1000


## 5. Build conditioning tensors and generate samples

**`mask_mode = "unconditional"`** — `cond_mask = 0` everywhere; the model receives no context and generates all OHLCV features from pure noise.

**`mask_mode = "predict_close"`** — Open, High, Low and Volume are loaded from `COND_DATA_DIR`, z-score normalise

In [5]:
# # Unconditional generation: no observed context
# observed_data = torch.zeros(N_SAMPLES, target_dim, seq_len, device=device)
# cond_mask     = torch.zeros(N_SAMPLES, target_dim, seq_len, device=device)

# # Normalised time positions in [0, 1], matching the training dataloader's index_norm mode
# observed_tp = torch.linspace(0.0, 1.0, seq_len, device=device).unsqueeze(0).expand(N_SAMPLES, -1)

# samples = processes.reverse_process(
#     model          = model,
#     shape          = (N_SAMPLES, target_dim, seq_len),
#     observed_data  = observed_data,
#     cond_mask      = cond_mask,
#     observed_tp    = observed_tp,
#     num_steps      = num_reverse_steps,
#     probability_flow = False,
#     device         = device,
# )  # → (N_SAMPLES, K, L)

# print(f"\nGenerated tensor shape : {samples.shape}")
# print(f"Mean  : {samples.mean().item():.4f}")
# print(f"Std   : {samples.std().item():.4f}")
# print(f"Range : [{samples.min().item():.4f}, {samples.max().item():.4f}]")

## 5. Generate samples

We generate unconditionally: `cond_mask = 0` everywhere so the model receives no context a

In [6]:
FEATURE_COLS = ["Open", "High", "Low", "Close", "Volume"]
CLOSE_IDX    = FEATURE_COLS.index("Close")               # 3
COND_INDICES = [i for i, f in enumerate(FEATURE_COLS) if f != "Close"]  # [0,1,2,4]

# Normalisation stats used to rescale conditioning data back to the original space
# after generation (filled in the predict_close branch; None in unconditional mode).
norm_stats = {}   # {sample_idx: {"mean": (K,), "std": (K,), "ticker": str}}

# ── Build observed_data and cond_mask ─────────────────────────────────────────
if mask_mode == "unconditional":
    observed_data = torch.zeros(N_SAMPLES, target_dim, seq_len, device=device)
    cond_mask     = torch.zeros(N_SAMPLES, target_dim, seq_len, device=device)
    cond_tickers  = [f"FAKE_{i+1:04d}" for i in range(N_SAMPLES)]

elif mask_mode == "predict_close":
    csv_files = sorted([f for f in os.listdir(COND_DATA_DIR) if f.endswith(".csv")])
    if len(csv_files) < N_SAMPLES:
        raise ValueError(
            f"Need {N_SAMPLES} CSV files but found only {len(csv_files)} in {COND_DATA_DIR}"
        )
    selected_files = csv_files[:N_SAMPLES]

    # Load optional pre-computed stats pickle
    precomp_stats = {}
    if COND_STATS_PATH is not None:
        with open(COND_STATS_PATH, "rb") as f:
            precomp_stats = pickle.load(f)

    obs_np    = np.zeros((N_SAMPLES, target_dim, seq_len), dtype=np.float32)
    cond_tickers = []

    for idx, fname in enumerate(selected_files):
        ticker = fname.split("_processed")[0]   # best-effort ticker name
        cond_tickers.append(ticker)

        df   = pd.read_csv(os.path.join(COND_DATA_DIR, fname))
        vals = df[FEATURE_COLS].values[:seq_len].T.astype(np.float32)  # (K, L)

        # z-score normalisation — use pre-computed stats if available, else per-series
        if ticker in precomp_stats:
            mu  = precomp_stats[ticker]["mean"].reshape(-1, 1)   # (K, 1)
            std = precomp_stats[ticker]["std"].reshape(-1, 1)
        else:
            mu  = vals.mean(axis=1, keepdims=True)
            std = vals.std(axis=1, keepdims=True).clip(min=1e-8)

        norm_stats[idx] = {"mean": mu.squeeze(), "std": std.squeeze(), "ticker": ticker}
        obs_np[idx] = (vals - mu) / std

    observed_data = torch.from_numpy(obs_np).to(device)

    # cond_mask: 1 = observed (O, H, L, V), 0 = to predict (Close)
    cond_mask = torch.zeros(N_SAMPLES, target_dim, seq_len, device=device)
    cond_mask[:, COND_INDICES, :] = 1.0

else:
    raise ValueError(f"Unknown mask_mode: {mask_mode!r}. Choose 'unconditional' or 'predict_close'.")

print(f"mask_mode     : {mask_mode}")
print(f"observed frac : {cond_mask.mean().item():.2f}  (fraction of observed values)")

mask_mode     : unconditional
observed frac : 0.00  (fraction of observed values)


In [ ]:
# ── Normalised time positions ─────────────────────────────────────────────────
observed_tp = torch.linspace(0.0, 1.0, seq_len, device=device).unsqueeze(0).expand(N_SAMPLES, -1)

# ── Run reverse diffusion ─────────────────────────────────────────────────────
samples = processes.reverse_process(
    model            = model,
    shape            = (N_SAMPLES, target_dim, seq_len),
    observed_data    = observed_data,
    cond_mask        = cond_mask,
    observed_tp      = observed_tp,
    num_steps        = num_reverse_steps,
    probability_flow = False,
    device           = device,
)  # → (N_SAMPLES, K, L)

print(f"\nGenerated tensor shape : {samples.shape}")
print(f"Mean  : {samples.mean().item():.4f}")
print(f"Std   : {samples.std().item():.4f}")
print(f"Range : [{samples.min().item():.4f}, {samples.max().item():.4f}]")

This is our SDE: <src.utils.WIP_SDE.VESDE object at 0x000001F75A1EED50>
This is the value of T: 1.0
Check prior ve: Mean = -0.0014333705184981227, Std = 0.9978894591331482
[reverse step 1/1000 | i=0 | t=1.0000]
mean=-0.0016, std=1.0037, min=-4.3129, max=4.1750
Time of last 100 steps: 3.4454891681671143. Time remaining 344.5489168167114.

[reverse step 2/1000 | i=1 | t=0.9990]
mean=-0.0021, std=1.0139, min=-4.4964, max=4.2690
Time of last 100 steps: 3.9191367626190186. Time remaining 391.91367626190186.

[reverse step 3/1000 | i=2 | t=0.9980]
mean=-0.0023, std=1.0240, min=-4.6244, max=4.2810
Time of last 100 steps: 3.5899851322174072. Time remaining 358.9985132217407.

[reverse step 4/1000 | i=3 | t=0.9970]
mean=-0.0023, std=1.0346, min=-4.6075, max=4.4682
Time of last 100 steps: 3.388566732406616. Time remaining 338.8566732406616.

[reverse step 5/1000 | i=4 | t=0.9960]
mean=-0.0026, std=1.0446, min=-4.7069, max=4.3584
Time of last 100 steps: 3.2479565143585205. Time remaining 324.7956

## 6. Save results to `data/generated/toy/`

**Unconditional mode** — each sample is saved as a separate CSV (`FAKE_XXXX_<start>_<end>_processed.csv`) with columns `Date, Open, High, Low, Close, Volume` (all model-generated).

**`predict_close` mode** — the CSV contains the *original* O, H, L, V values (from `COND_DATA_DIR`, denormalised) and the *model-predicted* Close (denormalised).  The output filename uses the source ticker name so results can be matched back to the input.

In both cases a companion `fake_stats_generated.pkl` stores per-ticker `{"mean": ..., "std": ...}` (in original d

In [ ]:
OUT_DIR = os.path.join("..", "data", "generated", "toy")
os.makedirs(OUT_DIR, exist_ok=True)

# Business-day date index of exactly seq_len days
dates = pd.bdate_range(start=START_DATE, end=END_DATE)
if len(dates) < seq_len:
    raise ValueError(
        f"Date range {START_DATE} → {END_DATE} yields only {len(dates)} business days "
        f"but seq_len = {seq_len}. Please extend END_DATE."
    )
dates = dates[:seq_len]

samples_np = samples.detach().cpu().numpy()   # (N_SAMPLES, K, L)
obs_np_cpu = observed_data.detach().cpu().numpy() if mask_mode == "predict_close" else None

stats_dict = {}

for i in range(N_SAMPLES):
    ticker = cond_tickers[i]
    gen    = samples_np[i]   # (K, L) — model output in normalised space

    if mask_mode == "unconditional":
        # All features are model-generated; no denormalisation needed
        series = gen
        mu  = series.mean(axis=1).astype(np.float32)
        std = series.std(axis=1).clip(min=1e-8).astype(np.float32)

    else:  # predict_close
        mu_arr  = norm_stats[i]["mean"]   # (K,)
        std_arr = norm_stats[i]["std"]    # (K,)

        # Denormalise full model output
        series_denorm = gen * std_arr[:, None] + mu_arr[:, None]   # (K, L)

        # Replace conditioned features with the original (denormalised) values
        orig = obs_np_cpu[i] * std_arr[:, None] + mu_arr[:, None]  # (K, L)
        series_denorm[COND_INDICES, :] = orig[COND_INDICES, :]

        series = series_denorm
        mu  = series.mean(axis=1).astype(np.float32)
        std = series.std(axis=1).clip(min=1e-8).astype(np.float32)

    df = pd.DataFrame(series.T, columns=FEATURE_COLS)   # (L, K)
    df.insert(0, "Date", dates.strftime("%d/%m/%Y"))

    suffix   = "predicted_close" if mask_mode == "predict_close" else "generated"
    filename = f"{ticker}_{START_DATE}_{END_DATE}_{suffix}.csv"
    df.to_csv(os.path.join(OUT_DIR, filename), index=False)

    stats_dict[ticker] = {"mean": mu, "std": std}

stats_path = os.path.join(OUT_DIR, "fake_stats_generated.pkl")
with open(stats_path, "wb") as f:
    pickle.dump(stats_dict, f)

print(f"mask_mode : {mask_mode}")
print(f"Saved {N_SAMPLES} CSV files  →  {OUT_DIR}")
print(f"Saved stats pickle          →  {stats_path}")

mask_mode : unconditional
Saved 1 CSV files  →  ..\data\generated\toy
Saved stats pickle          →  ..\data\generated\toy\fake_stats_generated.pkl
